In [ ]:
from sys import path

path.append("..")

import os
from pathlib import Path

os.chdir(Path.cwd().parent)

In [ ]:
import re
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import psutil

from graph_utils.reading import read_graph6
from descriptors.edge_descriptors import edge_descriptors_dict
from descriptors.node_descriptors import node_descriptors_dict
from comparison import tests
from graph_utils.reading import read_metadata



In [ ]:
for filename in os.listdir('raw_datasets'):
    if '.txt' in filename:
        old_filename = os.path.join('raw_datasets', filename)
        new_filename = re.sub(r'.txt', '', old_filename)
        print(old_filename, new_filename)

        os.rename(old_filename, new_filename)

In [ ]:
psutil.cpu_count()

## CHECK for desrciptors 
(edge and node should return the vector with the size respectfully proportionall to edges and nodes)

In [ ]:
gr = read_graph6('graph5')
for _ in range(1):
    g = next(gr)

for k, f in edge_descriptors_dict.items():
    print(k, f(g).shape)
print()
print()
print()

for k, f in node_descriptors_dict.items():
    print(k, f(g).shape)

In [ ]:
single_features = [(k, ) for k in edge_descriptors_dict.keys()]

node_features = [(f, ) for f in list(filter(lambda k: 'ldp' not in k, node_descriptors_dict.keys()))]
node_features.extend([tuple(filter(lambda k: 'ldp' in k and 'normalized' not in k, node_descriptors_dict.keys()))])
node_features.extend([tuple(filter(lambda k: 'ldp' in k and ('normalized' in k or 'degree' in k), node_descriptors_dict.keys()))])

single_features.extend(node_features)
single_features

In [ ]:
from graph_utils.reading import READ_PATH
import os

arguments_list = []

for dataset_name in  map(lambda x: '.'.join(x.split('.')[:-1]), filter(lambda x: '.g6' in x, os.listdir(READ_PATH))):
    # for features in [("moltop",), ("moltop","ltp"), ("moltop", "ldp"), ("moltop", "ltp", "ldp")]:
    for features in single_features:
    
        features = np.array(features)
        arguments_list.append(dict(dataset_name=dataset_name,
                                    features=features,))

In [ ]:
tests(arguments_list)

In [ ]:
metadata = read_metadata()

In [ ]:
from collections import Counter

def count_elements(row_string):
    c = Counter(row_string)
    return c[','] + 1 if ',' in c else 0
test_data = pd.read_parquet('processed_datasets/table.parquet')
test_data['count'] = test_data['result'].apply(count_elements)

test_data['percentage'] = test_data.apply(lambda x: x['count'] / metadata[x['dataset_name']]['graph_count'], axis=1)

In [ ]:
test_data = test_data.reset_index(drop=True)
test_data

In [ ]:

test_data['features'] = test_data['features'].astype('string').astype('category')
test_data.columns

In [ ]:
dataset_search = 'graph'

g = sns.pairplot(test_data[test_data['dataset_name'].map(lambda x: dataset_search in x)], 
             y_vars=['percentage'],
             x_vars=['features'],
             hue='dataset_name',
             palette='Spectral'
             )
fig = plt.gcf()
fig.set_size_inches(15,10)
plt.tight_layout()

for ax in g.axes.flatten():
    for label in ax.get_xticklabels():
        label.set_rotation(80)



In [ ]:
data2d = test_data.pivot(index="features", columns="dataset_name", values="percentage")

g = sns.heatmap(data=data2d, cmap='rocket_r')
fig = plt.gcf()
fig.set_size_inches(15,8)
plt.tight_layout()

g.set_yticks(np.arange(data2d.shape[0]) + 0.5)

_ = g.set_yticklabels(data2d.index)

g.set_xticks(np.arange(data2d.shape[1]) + 0.5)

_ = g.set_xticklabels(data2d.columns)
